# Cleanup Parquet files #
This jupyter notebook is to be run after running [create_parquet_files.ipynb](create_parquet_files.ipynb)

In [ ]:
import duckdb
import pandas as pd

con = duckdb.connect()

# 2021 Data #

In [ ]:
# -------------------------------------------------------------------
# 2021 Census
# -------------------------------------------------------------------

input_file = "temp_parquet_data/2021_raw_census_data.parquet"

# -------------------------------------------------------------------
# Main data
# -------------------------------------------------------------------

con.execute(f"""
    COPY (
        SELECT DISTINCT
            CENSUS_YEAR,
            DGUID,
            CHARACTERISTIC_ID,
            C1_COUNT_TOTAL,
            "C2_COUNT_MEN+",
            "C3_COUNT_WOMEN+",
            C10_RATE_TOTAL,
            "C11_RATE_MEN+",
            "C12_RATE_WOMEN+"
        FROM '{input_file}'
    )
    TO 'parquet_data/2021_census_data.parquet'
    (FORMAT PARQUET, COMPRESSION ZSTD);
""")

# -------------------------------------------------------------------
# Characteristics lookup table
# -------------------------------------------------------------------

con.execute(f"""
    COPY (
        SELECT DISTINCT
            CENSUS_YEAR,
            CHARACTERISTIC_ID,
            CHARACTERISTIC_NAME
        FROM '{input_file}'
        ORDER BY CHARACTERISTIC_ID
    )
    TO 'parquet_data/2021_characteristics.parquet'
    (FORMAT PARQUET, COMPRESSION ZSTD);
""")

# -------------------------------------------------------------------
# Geography lookup table
# -------------------------------------------------------------------

con.execute(f"""
    COPY (
        SELECT DISTINCT
            CENSUS_YEAR,
            DGUID,
            GEO_LEVEL,
            GEO_NAME
        FROM '{input_file}'
        ORDER BY DGUID
    )
    TO 'parquet_data/2021_geographies.parquet'
    (FORMAT PARQUET, COMPRESSION ZSTD);
""")

# -------------------------------------------------------------------
# List of data columns
# -------------------------------------------------------------------

con.execute("DROP TABLE IF EXISTS column_values_2021;")

con.execute("""
    CREATE TABLE column_values_2021 (
        column_id INTEGER PRIMARY KEY,
        column_name VARCHAR
    );
""")

con.execute("""
    INSERT INTO column_values_2021 VALUES
        (1, 'C1_COUNT_TOTAL'),
        (2, 'C2_COUNT_MEN+'),
        (3, 'C3_COUNT_WOMEN+'),
        (4, 'C10_RATE_TOTAL'),
        (5, 'C11_RATE_MEN+'),
        (6, 'C12_RATE_WOMEN+');
""")

# -------------------------------------------------------------------
# Characteristic -> column relationship
# -------------------------------------------------------------------

con.execute("DROP TABLE IF EXISTS characteristic_columns_2021;")

con.execute("""
    CREATE TABLE characteristic_columns_2021 (
        CENSUS_YEAR INTEGER,
        CHARACTERISTIC_ID INTEGER,
        column_id INTEGER
    );
""")

columns = [
    "C1_COUNT_TOTAL",
    "C2_COUNT_MEN+",
    "C3_COUNT_WOMEN+",
    "C10_RATE_TOTAL",
    "C11_RATE_MEN+",
    "C12_RATE_WOMEN+"
]

for i, column in enumerate(columns, start=1):
    con.execute(f"""
        INSERT INTO characteristic_columns_2021
        SELECT DISTINCT
            CENSUS_YEAR,
            CHARACTERISTIC_ID,
            {i}
        FROM '{input_file}'
        WHERE "{column}" IS NOT NULL;
    """)

# -------------------------------------------------------------------
# Save lookup tables
# -------------------------------------------------------------------

con.execute("""
    COPY column_values_2021
    TO 'parquet_data/2021_columns.parquet'
    (FORMAT PARQUET, COMPRESSION ZSTD);
""")

con.execute("""
    COPY characteristic_columns_2021
    TO 'parquet_data/2021_characteristic_columns.parquet'
    (FORMAT PARQUET, COMPRESSION ZSTD);
""")

# 2016 Data #

In [ ]:
# -------------------------------------------------------------------
# 2016 Census
# -------------------------------------------------------------------

input_file = "temp_parquet_data/2016_raw_census_data.parquet"

# -------------------------------------------------------------------
# Main data
# -------------------------------------------------------------------

con.execute(f"""
    COPY (
        SELECT DISTINCT
            CENSUS_YEAR,
            GEO_CODE,
            CHARACTERISTIC_ID,
            COUNT_TOTAL,
            COUNT_MALE,
            COUNT_FEMALE
        FROM '{input_file}'
    )
    TO 'parquet_data/2016_census_data.parquet'
    (FORMAT PARQUET, COMPRESSION ZSTD);
""")

# -------------------------------------------------------------------
# Characteristics lookup table
# -------------------------------------------------------------------

con.execute(f"""
    COPY (
        SELECT DISTINCT
            CENSUS_YEAR,
            CHARACTERISTIC_ID,
            CHARACTERISTIC_NAME
        FROM '{input_file}'
        ORDER BY CHARACTERISTIC_ID
    )
    TO 'parquet_data/2016_characteristics.parquet'
    (FORMAT PARQUET, COMPRESSION ZSTD);
""")

# -------------------------------------------------------------------
# Geography lookup table
# -------------------------------------------------------------------

con.execute(f"""
    COPY (
        SELECT DISTINCT
            CENSUS_YEAR,
            GEO_CODE,
            GEO_LEVEL,
            GEO_NAME
        FROM '{input_file}'
        ORDER BY GEO_CODE
    )
    TO 'parquet_data/2016_geographies.parquet'
    (FORMAT PARQUET, COMPRESSION ZSTD);
""")

# -------------------------------------------------------------------
# List of data columns
# -------------------------------------------------------------------

con.execute("DROP TABLE IF EXISTS column_values_2016;")

con.execute("""
    CREATE TABLE column_values_2016 (
        column_id INTEGER PRIMARY KEY,
        column_name VARCHAR
    );
""")

con.execute("""
    INSERT INTO column_values_2016 VALUES
        (1, 'COUNT_TOTAL'),
        (2, 'COUNT_MALE'),
        (3, 'COUNT_FEMALE');
""")

# -------------------------------------------------------------------
# Characteristic -> column relationship
# -------------------------------------------------------------------

con.execute("DROP TABLE IF EXISTS characteristic_columns_2016;")

con.execute("""
    CREATE TABLE characteristic_columns_2016 (
        CENSUS_YEAR INTEGER,
        CHARACTERISTIC_ID INTEGER,
        column_id INTEGER
    );
""")

columns = [
    "COUNT_TOTAL",
    "COUNT_MALE",
    "COUNT_FEMALE",
]

for i, column in enumerate(columns, start=1):
    con.execute(f"""
        INSERT INTO characteristic_columns_2016
        SELECT DISTINCT
            CENSUS_YEAR,
            CHARACTERISTIC_ID,
            {i}
        FROM '{input_file}'
        WHERE "{column}" IS NOT NULL;
    """)

# -------------------------------------------------------------------
# Save lookup tables
# -------------------------------------------------------------------

con.execute("""
    COPY column_values_2016
    TO 'parquet_data/2016_columns.parquet'
    (FORMAT PARQUET, COMPRESSION ZSTD);
""")

con.execute("""
    COPY characteristic_columns_2016
    TO 'parquet_data/2016_characteristic_columns.parquet'
    (FORMAT PARQUET, COMPRESSION ZSTD);
""")

# 2011 Data #

In [ ]:
# -------------------------------------------------------------------
# 2011 Census / NHS
# -------------------------------------------------------------------

input_file1 = "temp_parquet_data/2011_raw_census_data.parquet"
input_file2 = "temp_parquet_data/2011_raw_nhs_data.parquet"
input_file3 = "temp_parquet_data/2011_raw_nhs_da.parquet"

# -------------------------------------------------------------------
# Main data
# -------------------------------------------------------------------

con.execute("DROP TABLE IF EXISTS DATA_2011;")

con.execute("""
    CREATE TABLE DATA_2011 (
        GEO_CODE VARCHAR,
        CHARACTERISTIC_ID INTEGER,
        COUNT_TOTAL DOUBLE,
        COUNT_MALE DOUBLE,
        COUNT_FEMALE DOUBLE
    );
""")

# 2011 Census
con.execute(f"""
    INSERT INTO DATA_2011
    SELECT DISTINCT
        GEO_CODE,
        CHARACTERISTIC_ID,
        COUNT_TOTAL,
        COUNT_MALE,
        COUNT_FEMALE
    FROM '{input_file1}';
""")

# 2011 NHS
con.execute(f"""
    INSERT INTO DATA_2011
    SELECT DISTINCT
        GEO_CODE,
        CHARACTERISTIC_ID,
        COUNT_TOTAL,
        COUNT_MALE,
        COUNT_FEMALE
    FROM '{input_file2}';
""")

# -------------------------------------------------------------------
# 2011 NHS DA
# -------------------------------------------------------------------

con.execute("DROP TABLE IF EXISTS DA_NHS_DATA_2011;")

con.execute("""
    CREATE TABLE DA_NHS_DATA_2011 (
        GEO_CODE VARCHAR,
        CHARACTERISTIC_ID INTEGER,
        COUNT_TOTAL DOUBLE
    );
""")

con.execute(f"""
    INSERT INTO DA_NHS_DATA_2011
    SELECT DISTINCT
        GEO_CODE,
        CHARACTERISTIC_ID,
        COUNT_TOTAL
    FROM '{input_file3}';
""")

# -------------------------------------------------------------------
# List of data columns
# -------------------------------------------------------------------

con.execute("DROP TABLE IF EXISTS column_values_2011;")

con.execute("""
    CREATE TABLE column_values_2011 (
        column_id INTEGER PRIMARY KEY,
        column_name VARCHAR
    );
""")

con.execute("""
    INSERT INTO column_values_2011 VALUES
        (1, 'COUNT_TOTAL'),
        (2, 'COUNT_MALE'),
        (3, 'COUNT_FEMALE');
""")

# -------------------------------------------------------------------
# Characteristic -> column relationship
# -------------------------------------------------------------------

con.execute("DROP TABLE IF EXISTS characteristic_columns_2011;")

con.execute("""
    CREATE TABLE characteristic_columns_2011 (
        CHARACTERISTIC_ID INTEGER,
        column_id INTEGER
    );
""")

columns = [
    "COUNT_TOTAL",
    "COUNT_MALE",
    "COUNT_FEMALE",
]

for i, column in enumerate(columns, start=1):
    con.execute(f"""
        INSERT INTO characteristic_columns_2011
        SELECT DISTINCT
            CHARACTERISTIC_ID,
            {i}
        FROM read_parquet(['{input_file2}', '{input_file1}'])
        WHERE "{column}" IS NOT NULL;
    """)

# -------------------------------------------------------------------
# Characteristics lookup table
# -------------------------------------------------------------------

df = pd.read_csv(
    "characteristics/2011_characteristics.csv",
    usecols=["CHARACTERISTIC_ID", "CHARACTERISTIC_NAME"]
)

df.to_parquet(
    "parquet_data/2011_characteristics.parquet",
    index=False
)

# -------------------------------------------------------------------
# NHS DA characteristics lookup table
# -------------------------------------------------------------------

con.execute("DROP TABLE IF EXISTS DA_CHAR_2011;")

con.execute("""
    CREATE TABLE DA_CHAR_2011 (
        CHARACTERISTIC_ID INTEGER,
        CHARACTERISTIC_NAME VARCHAR
    );
""")

con.execute(f"""
    INSERT INTO DA_CHAR_2011
    SELECT DISTINCT
        CHARACTERISTIC_ID,
        CHARACTERISTIC_NAME
    FROM '{input_file3}'
    ORDER BY CHARACTERISTIC_ID;
""")

# -------------------------------------------------------------------
# Save everything
# -------------------------------------------------------------------

con.execute("""
    COPY DATA_2011
    TO 'parquet_data/2011_census_data.parquet'
    (FORMAT PARQUET, COMPRESSION ZSTD);
""")

con.execute("""
    COPY column_values_2011
    TO 'parquet_data/2011_columns.parquet'
    (FORMAT PARQUET, COMPRESSION ZSTD);
""")

con.execute("""
    COPY characteristic_columns_2011
    TO 'parquet_data/2011_characteristic_columns.parquet'
    (FORMAT PARQUET, COMPRESSION ZSTD);
""")

con.execute("""
    COPY DA_NHS_DATA_2011
    TO 'parquet_data/2011_nhs_da_data.parquet'
    (FORMAT PARQUET, COMPRESSION ZSTD);
""")

con.execute("""
    COPY DA_CHAR_2011
    TO 'parquet_data/2011_nhs_da_characteristics.parquet'
    (FORMAT PARQUET, COMPRESSION ZSTD);
""")

# 2006 Data #

In [ ]:
# -------------------------------------------------------------------
# 2006 Census / NHS
# -------------------------------------------------------------------

input_file = "temp_parquet_data/2006_raw_census_data.parquet"

# -------------------------------------------------------------------
# Main data
# -------------------------------------------------------------------

con.execute("DROP TABLE IF EXISTS DATA_2006;")

con.execute("""
    CREATE TABLE DATA_2006 (
        GEO_CODE VARCHAR,
        CHARACTERISTIC_ID INTEGER,
        CENSUS_YEAR INTEGER,
        COUNT_TOTAL DOUBLE
    );
""")

# 2006 Census
con.execute(f"""
    INSERT INTO DATA_2006
    SELECT DISTINCT
        GEO_CODE,
        CHARACTERISTIC_ID,
        CENSUS_YEAR,
        COUNT_TOTAL
    FROM '{input_file}';
""")

# -------------------------------------------------------------------
# Characteristics lookup table
# -------------------------------------------------------------------

df = pd.read_csv(
    "characteristics/2006_characteristics.csv",
    usecols=["CHARACTERISTIC_ID", "CHARACTERISTIC_NAME"]
)

df.to_parquet(
    "parquet_data/2006_characteristics.parquet",
    index=False
)

# 2001 Data #

In [ ]:
# -------------------------------------------------------------------
# 2001 Census
# -------------------------------------------------------------------

input_file = "temp_parquet_data/2001_raw_census_data.parquet"

# -------------------------------------------------------------------
# Main data
# -------------------------------------------------------------------

con.execute("DROP TABLE IF EXISTS DATA_2001;")

con.execute("""
    CREATE TABLE DATA_2001 (
        GEO_CODE VARCHAR,
        CHARACTERISTIC_ID INTEGER,
        CENSUS_YEAR INTEGER,
        COUNT_TOTAL DOUBLE
    );
""")

# 2001 Census
con.execute(f"""
    INSERT INTO DATA_2001
    SELECT DISTINCT
        GEO_CODE,
        CHARACTERISTIC_ID,
        CENSUS_YEAR,
        COUNT_TOTAL
    FROM '{input_file}';
""")

# -------------------------------------------------------------------
# Characteristics lookup table
# -------------------------------------------------------------------

df = pd.read_csv(
    "characteristics/2001_characteristics.csv",
    usecols=["CHARACTERISTIC_ID", "CHARACTERISTIC_NAME"]
)

df.to_parquet(
    "parquet_data/2001_characteristics.parquet",
    index=False
)

# 1996 Data #

In [ ]:
# -------------------------------------------------------------------
# 1996 Census
# -------------------------------------------------------------------

input_file = "temp_parquet_data/1996_raw_census_data.parquet"

# -------------------------------------------------------------------
# Main data
# -------------------------------------------------------------------

con.execute("DROP TABLE IF EXISTS DATA_1996;")

con.execute("""
    CREATE TABLE DATA_1996 (
        GEO_CODE VARCHAR,
        CHARACTERISTIC_ID INTEGER,
        CENSUS_YEAR INTEGER,
        COUNT_TOTAL DOUBLE
    );
""")

# 1996 Census
con.execute(f"""
    INSERT INTO DATA_1996
    SELECT DISTINCT
        GEO_CODE,
        CHARACTERISTIC_ID,
        CENSUS_YEAR,
        COUNT_TOTAL
    FROM '{input_file}';
""")

# -------------------------------------------------------------------
# Characteristics lookup table
# -------------------------------------------------------------------

df = pd.read_csv(
    "characteristics/1996_characteristics.csv",
    usecols=["CHARACTERISTIC_ID", "CHARACTERISTIC_NAME"]
)

df.to_parquet(
    "parquet_data/1996_characteristics.parquet",
    index=False
)

# 1991 Data #

In [ ]:
# -------------------------------------------------------------------
# 1991 Census
# -------------------------------------------------------------------

input_file = "temp_parquet_data/1991_raw_census_data.parquet"

# -------------------------------------------------------------------
# Main data
# -------------------------------------------------------------------

con.execute("DROP TABLE IF EXISTS DATA_1991;")

con.execute("""
    CREATE TABLE DATA_1991 (
        GEO_CODE VARCHAR,
        CHARACTERISTIC_ID INTEGER,
        COUNT_TOTAL DOUBLE
    );
""")

# 1991 Census
con.execute(f"""
    INSERT INTO DATA_1991
    SELECT DISTINCT
        GEO_CODE,
        CHARACTERISTIC_ID,
        COUNT_TOTAL
    FROM '{input_file}';
""")

# -------------------------------------------------------------------
# Characteristics lookup table
# -------------------------------------------------------------------

df = pd.read_csv(
    "characteristics/1991_characteristics.csv",
    usecols=["CHARACTERISTIC_ID", "CHARACTERISTIC_NAME"]
)

df.to_parquet(
    "parquet_data/1991_characteristics.parquet",
    index=False
)

In [ ]:
# -------------------------------------------------------------------
# 1986 Census
# -------------------------------------------------------------------

input_file = "temp_parquet_data/1986_raw_census_data.parquet"

# -------------------------------------------------------------------
# Main data
# -------------------------------------------------------------------

con.execute("DROP TABLE IF EXISTS DATA_1986;")

con.execute("""
    CREATE TABLE DATA_1986 (
        GEO_CODE VARCHAR,
        CHARACTERISTIC_ID INTEGER,
        COUNT_TOTAL DOUBLE
    );
""")

# 1986 Census
con.execute(f"""
    INSERT INTO DATA_1986
    SELECT DISTINCT
        GEO_CODE,
        CHARACTERISTIC_ID,
        COUNT_TOTAL
    FROM '{input_file}';
""")

# -------------------------------------------------------------------
# Characteristics lookup table
# -------------------------------------------------------------------

# 1986 data already has consistent characteristics
con.execute(f"""
    COPY (
        SELECT DISTINCT
            CHARACTERISTIC_ID,
            CHARACTERISTIC_NAME
        FROM '{input_file}'
        ORDER BY CHARACTERISTIC_ID
    )
    TO 'parquet_data/1986_characteristics.parquet'
    (FORMAT PARQUET, COMPRESSION ZSTD);
""")

# 1981 Data #

In [ ]:
# -------------------------------------------------------------------
# 1981 Census
# -------------------------------------------------------------------

input_file = "temp_parquet_data/1981_raw_census_data.parquet"

# -------------------------------------------------------------------
# Main data
# -------------------------------------------------------------------

con.execute("DROP TABLE IF EXISTS DATA_1981;")

con.execute("""
    CREATE TABLE DATA_1981 (
        GEO_CODE VARCHAR,
        CHARACTERISTIC_ID INTEGER,
        COUNT_TOTAL DOUBLE
    );
""")

# 1981 Census
con.execute(f"""
    INSERT INTO DATA_1981
    SELECT DISTINCT
        GEO_CODE,
        CHARACTERISTIC_ID,
        COUNT_TOTAL
    FROM '{input_file}';
""")

# -------------------------------------------------------------------
# Characteristics lookup table
# -------------------------------------------------------------------

df = pd.read_csv(
    "characteristics/1981_characteristics.csv",
    usecols=["CHARACTERISTIC_ID", "CHARACTERISTIC_NAME"]
)

df.to_parquet(
    "parquet_data/1981_characteristics.parquet",
    index=False
)

# Run this to close the connection #

In [ ]:
con.close()